In [0]:

from pyspark.sql import functions as F

fact = spark.table(
    "proyecto_bigdata.clinica_levante_lazaro.fact_citas"
)

tiempo = spark.table(
    "proyecto_bigdata.clinica_levante_lazaro.dim_tiempo"
)

print("Registros en fact_citas:", fact.count())
print("Registros en dim_tiempo:", tiempo.count())



Registros en fact_citas: 98033
Registros en dim_tiempo: 730


In [0]:
datos_experimento = (
    fact.join(tiempo, on="id_tiempo", how="inner")
    .select(
        fact["*"],
        tiempo["fecha"],
        tiempo["anio"],
        tiempo["mes"]
    )
)

print("Registros del experimento:", datos_experimento.count())

display(datos_experimento.limit(10))

Registros del experimento: 98033


id_hecho,id_cita,id_tiempo,id_paciente,id_medico,id_especialidad,id_estado,cantidad_citas,duracion_min,costo_consulta,fecha,anio,mes
1,1,723,3851,10,3,2,1,35,150.00,2026-08-21,2026,8
2,2,600,7536,13,1,2,1,40,140.00,2026-08-19,2026,8
3,3,332,7536,11,4,2,1,20,80.00,2026-08-14,2026,8
4,4,662,10484,11,4,3,1,15,80.00,2026-08-02,2026,8
5,5,672,1570,1,2,2,1,40,100.00,2026-09-19,2026,9
6,6,44,6764,11,4,2,1,25,80.00,2026-07-18,2026,7
7,7,229,3851,5,5,3,1,40,130.00,2026-09-04,2026,9
8,8,416,1570,13,1,3,1,40,140.00,2026-07-04,2026,7
9,9,458,7536,10,3,2,1,35,150.00,2026-09-23,2026,9
10,10,580,6764,13,1,2,1,50,140.00,2026-08-22,2026,8


Databricks visualization. Run in Databricks to view.

In [0]:

# Escenario A: tabla Delta sin particionar
(
    datos_experimento.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "proyecto_bigdata.clinica_levante_lazaro.exp_sin_particion"
    )
)

# Escenario B: tabla Delta particionada por año y mes
(
    datos_experimento.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("anio", "mes")
    .saveAsTable(
        "proyecto_bigdata.clinica_levante_lazaro.exp_particionada"
    )
)

print("Tablas experimentales creadas.")


Tablas experimentales creadas.


In [0]:

tabla_a = spark.table(
    "proyecto_bigdata.clinica_levante_lazaro.exp_sin_particion"
)

tabla_b = spark.table(
    "proyecto_bigdata.clinica_levante_lazaro.exp_particionada"
)

print("Registros sin particionar:", tabla_a.count())
print("Registros particionados:", tabla_b.count())

print("Diferencias entre tablas:")
print("A menos B:", tabla_a.exceptAll(tabla_b).count())
print("B menos A:", tabla_b.exceptAll(tabla_a).count())


Registros sin particionar: 98033
Registros particionados: 98033
Diferencias entre tablas:
A menos B: 0
B menos A: 0


In [0]:

display(
    tabla_a.groupBy("anio", "mes")
    .count()
    .orderBy("anio", "mes")
)


anio,mes,count
2025,1,4239
2025,2,3838
2025,3,4107
2025,4,4141
2025,5,4196
2025,6,4007
2025,7,4122
2025,8,4204
2025,9,3966
2025,10,4187


Databricks visualization. Run in Databricks to view.

In [0]:


import time

# Reemplaza estos valores por un período existente
ANIO = 2026
MES = 11

def medir_consulta(nombre_tabla):
    consulta = f"""
        SELECT
            COUNT(*) AS total_citas,
            AVG(duracion_min) AS duracion_promedio,
            SUM(costo_consulta) AS costo_total
        FROM {nombre_tabla}
        WHERE anio = {ANIO}
          AND mes = {MES}
    """

    inicio = time.perf_counter()
    resultado = spark.sql(consulta).collect()
    fin = time.perf_counter()

    return fin - inicio, resultado

# Calentamiento: no guardamos estos tiempos
medir_consulta(
    "proyecto_bigdata.clinica_levante_lazaro.exp_sin_particion"
)
medir_consulta(
    "proyecto_bigdata.clinica_levante_lazaro.exp_particionada"
)

# Tres repeticiones por escenario
tiempos_a = []
tiempos_b = []

for i in range(3):
    tiempo_a, resultado_a = medir_consulta(
        "proyecto_bigdata.clinica_levante_lazaro.exp_sin_particion"
    )

    tiempo_b, resultado_b = medir_consulta(
        "proyecto_bigdata.clinica_levante_lazaro.exp_particionada"
    )

    tiempos_a.append(tiempo_a)
    tiempos_b.append(tiempo_b)

print("Tiempos sin particionar:", tiempos_a)
print("Tiempos particionados:", tiempos_b)

print("Promedio sin particionar:", sum(tiempos_a) / len(tiempos_a))
print("Promedio particionado:", sum(tiempos_b) / len(tiempos_b))

print("Resultado sin particionar:", resultado_a)
print("Resultado particionado:", resultado_b)



Tiempos sin particionar: [0.8698834299999589, 0.6429708249999067, 0.6645249770000419]
Tiempos particionados: [0.5847317350001049, 0.5713290349999625, 0.5631830710000258]
Promedio sin particionar: 0.7257930773333024
Promedio particionado: 0.5730812803333644
Resultado sin particionar: [Row(total_citas=3980, duracion_promedio=34.697236180904525, costo_total=Decimal('468520.00'))]
Resultado particionado: [Row(total_citas=3980, duracion_promedio=34.697236180904525, costo_total=Decimal('468520.00'))]


In [0]:

spark.sql("""
    DESCRIBE DETAIL
    proyecto_bigdata.clinica_levante_lazaro.exp_sin_particion
""").select(
    "format", "numFiles", "sizeInBytes", "partitionColumns"
).show(truncate=False)

spark.sql("""
    DESCRIBE DETAIL
    proyecto_bigdata.clinica_levante_lazaro.exp_particionada
""").select(
    "format", "numFiles", "sizeInBytes", "partitionColumns"
).show(truncate=False)



+------+--------+-----------+----------------+
|format|numFiles|sizeInBytes|partitionColumns|
+------+--------+-----------+----------------+
|delta |1       |678033     |[]              |
+------+--------+-----------+----------------+

+------+--------+-----------+----------------+
|format|numFiles|sizeInBytes|partitionColumns|
+------+--------+-----------+----------------+
|delta |24      |745337     |[anio, mes]     |
+------+--------+-----------+----------------+



In [0]:
datos_experimento = (
    fact.join(tiempo, on="id_tiempo", how="inner")
    .select(
        fact["*"],
        tiempo["fecha"],
        tiempo["anio"],
        tiempo["mes"]
    )
)

print("Registros del experimento:", datos_experimento.count())

display(datos_experimento.limit(10))

Registros del experimento: 98033


id_hecho,id_cita,id_tiempo,id_paciente,id_medico,id_especialidad,id_estado,cantidad_citas,duracion_min,costo_consulta,fecha,anio,mes
1,1,723,3851,10,3,2,1,35,150.00,2026-08-21,2026,8
2,2,600,7536,13,1,2,1,40,140.00,2026-08-19,2026,8
3,3,332,7536,11,4,2,1,20,80.00,2026-08-14,2026,8
4,4,662,10484,11,4,3,1,15,80.00,2026-08-02,2026,8
5,5,672,1570,1,2,2,1,40,100.00,2026-09-19,2026,9
6,6,44,6764,11,4,2,1,25,80.00,2026-07-18,2026,7
7,7,229,3851,5,5,3,1,40,130.00,2026-09-04,2026,9
8,8,416,1570,13,1,3,1,40,140.00,2026-07-04,2026,7
9,9,458,7536,10,3,2,1,35,150.00,2026-09-23,2026,9
10,10,580,6764,13,1,2,1,50,140.00,2026-08-22,2026,8


Databricks visualization. Run in Databricks to view.

In [0]:
# ESCENARIO A: Delta sin particionar
(
    datos_experimento.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "proyecto_bigdata.clinica_levante_lazaro.exp_sin_particion"
    )
)

# ESCENARIO B: Delta particionada por año y mes
(
    datos_experimento.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("anio", "mes")
    .saveAsTable(
        "proyecto_bigdata.clinica_levante_lazaro.exp_particionada"
    )
)

print("Tablas experimentales creadas correctamente.")

Tablas experimentales creadas correctamente.


In [0]:
tabla_a = spark.table(
    "proyecto_bigdata.clinica_levante_lazaro.exp_sin_particion"
)

tabla_b = spark.table(
    "proyecto_bigdata.clinica_levante_lazaro.exp_particionada"
)

print("Registros sin particionar:", tabla_a.count())
print("Registros particionados:", tabla_b.count())

print("Diferencias entre tablas:")
print("A menos B:", tabla_a.exceptAll(tabla_b).count())
print("B menos A:", tabla_b.exceptAll(tabla_a).count())

Registros sin particionar: 98033
Registros particionados: 98033
Diferencias entre tablas:
A menos B: 0
B menos A: 0


In [0]:
display(
    tabla_a.groupBy("anio", "mes")
    .count()
    .orderBy("anio", "mes")
)

anio,mes,count
2025,1,4239
2025,2,3838
2025,3,4107
2025,4,4141
2025,5,4196
2025,6,4007
2025,7,4122
2025,8,4204
2025,9,3966
2025,10,4187


In [0]:

import time

ANIO = 2025
MES = 1

def medir_consulta(nombre_tabla):
    consulta = f"""
        SELECT
            COUNT(*) AS total_citas,
            AVG(duracion_min) AS duracion_promedio,
            SUM(costo_consulta) AS costo_total
        FROM {nombre_tabla}
        WHERE anio = {ANIO}
          AND mes = {MES}
    """

    inicio = time.perf_counter()
    resultado = spark.sql(consulta).collect()
    fin = time.perf_counter()

    return fin - inicio, resultado

# Calentamiento de ambas tablas
medir_consulta(
    "proyecto_bigdata.clinica_levante_lazaro.exp_sin_particion"
)

medir_consulta(
    "proyecto_bigdata.clinica_levante_lazaro.exp_particionada"
)

# Tres mediciones por escenario
tiempos_a = []
tiempos_b = []

for i in range(3):
    tiempo_a, resultado_a = medir_consulta(
        "proyecto_bigdata.clinica_levante_lazaro.exp_sin_particion"
    )

    tiempo_b, resultado_b = medir_consulta(
        "proyecto_bigdata.clinica_levante_lazaro.exp_particionada"
    )

    tiempos_a.append(tiempo_a)
    tiempos_b.append(tiempo_b)

print("Tiempos sin particionar:", tiempos_a)
print("Tiempos particionados:", tiempos_b)

print("Promedio sin particionar:", sum(tiempos_a) / 3)
print("Promedio particionado:", sum(tiempos_b) / 3)

print("Resultado sin particionar:", resultado_a)
print("Resultado particionado:", resultado_b)


Tiempos sin particionar: [0.6520704309999701, 0.6378287740000133, 0.6370687859999862]
Tiempos particionados: [0.5686308279999821, 0.5820261549999941, 0.5463058240000009]
Promedio sin particionar: 0.6423226636666565
Promedio particionado: 0.5656542689999924
Resultado sin particionar: [Row(total_citas=4239, duracion_promedio=34.77824958716678, costo_total=Decimal('501420.00'))]
Resultado particionado: [Row(total_citas=4239, duracion_promedio=34.77824958716678, costo_total=Decimal('501420.00'))]


In [0]:
tabla_sin_particion = spark.sql("""
    DESCRIBE DETAIL
    proyecto_bigdata.clinica_levante_lazaro.exp_sin_particion
""")

tabla_particionada = spark.sql("""
    DESCRIBE DETAIL
    proyecto_bigdata.clinica_levante_lazaro.exp_particionada
""")

print("ESCENARIO A: SIN PARTICIONAR")
tabla_sin_particion.select(
    "numFiles",
    "sizeInBytes",
    "partitionColumns"
).show(truncate=False)

print("ESCENARIO B: PARTICIONADA")
tabla_particionada.select(
    "numFiles",
    "sizeInBytes",
    "partitionColumns"
).show(truncate=False)

ESCENARIO A: SIN PARTICIONAR
+--------+-----------+----------------+
|numFiles|sizeInBytes|partitionColumns|
+--------+-----------+----------------+
|1       |678033     |[]              |
+--------+-----------+----------------+

ESCENARIO B: PARTICIONADA
+--------+-----------+----------------+
|numFiles|sizeInBytes|partitionColumns|
+--------+-----------+----------------+
|24      |745337     |[anio, mes]     |
+--------+-----------+----------------+



In [0]:

import time
import statistics

nombre_a = "proyecto_bigdata.clinica_levante_lazaro.exp_sin_particion"
nombre_b = "proyecto_bigdata.clinica_levante_lazaro.exp_particionada"

def medir_tiempo(nombre_tabla):
    consulta = f"""
        SELECT
            COUNT(*) AS total_citas,
            AVG(duracion_min) AS duracion_promedio,
            SUM(costo_consulta) AS costo_total
        FROM {nombre_tabla}
        WHERE anio = 2025 AND mes = 1
    """

    inicio = time.perf_counter()
    resultado = spark.sql(consulta).collect()
    fin = time.perf_counter()

    return fin - inicio, resultado

# Calentamiento de ambas tablas
medir_tiempo(nombre_a)
medir_tiempo(nombre_b)

tiempos_a = []
tiempos_b = []

# Cinco repeticiones alternando el orden
for i in range(5):
    if i % 2 == 0:
        tiempo_a, resultado_a = medir_tiempo(nombre_a)
        tiempo_b, resultado_b = medir_tiempo(nombre_b)
    else:
        tiempo_b, resultado_b = medir_tiempo(nombre_b)
        tiempo_a, resultado_a = medir_tiempo(nombre_a)

    tiempos_a.append(tiempo_a)
    tiempos_b.append(tiempo_b)

print("Tiempos A:", tiempos_a)
print("Tiempos B:", tiempos_b)

print("Promedio A:", statistics.mean(tiempos_a))
print("Promedio B:", statistics.mean(tiempos_b))

print("Mediana A:", statistics.median(tiempos_a))
print("Mediana B:", statistics.median(tiempos_b))

print("Resultado A:", resultado_a)
print("Resultado B:", resultado_b)



Tiempos A: [0.5616854110000986, 0.5893339260001085, 0.6203754010000466, 0.701752112000122, 0.575261318999992]
Tiempos B: [0.5606450279999535, 0.6310263830000622, 0.5641535510001177, 0.5893450470000516, 0.615673981999862]
Promedio A: 0.6096816338000736
Promedio B: 0.5921687982000094
Mediana A: 0.5893339260001085
Mediana B: 0.5893450470000516
Resultado A: [Row(total_citas=4239, duracion_promedio=34.77824958716678, costo_total=Decimal('501420.00'))]
Resultado B: [Row(total_citas=4239, duracion_promedio=34.77824958716678, costo_total=Decimal('501420.00'))]
